# Which resistor can carry its current through another path?

A short circuit diagram leads to a matrix that describes both voltage differences and sensitivity to losing an edge. Change one conductance and follow where the current goes.

## Ground a node and derive the electrical leverage

Nodes A, B and C form a triangular loop. Node D connects only to C. Every edge is an ideal linear resistor. Orient an edge from node $i$ to node $j$ and give it an incidence vector $b_e$ with a +1 at $i$ and a −1 at $j$. Its current under nodal voltages $v$ is $g_e b_e^T v$, where conductance $g_e$ is measured in siemens. Kirchhoff's current law sums all outgoing currents into the net injection $f$. It gives

$$Lv=f,\qquad L=\sum_e g_e b_eb_e^T.$$

Adding a constant to every potential preserves all currents. Fix the potential of D at zero and remove its row and column. For a connected network the resulting grounded matrix is positive definite. Inject one ampere at one end of an edge and withdraw it at the other. With grounded coordinates the voltage difference per unit current is the effective resistance $R_e=b_e^T L_g^{-1} b_e$. The fraction of this test current that flows through the tested edge is $\ell_e=g_eR_e$. A bridge carries the full ampere. A parallel path reduces its share.

There is also a global check on the calculation. The sum of all the leverages is $\mathrm{tr}(L_g^{-1}\sum_e g_e b_e b_e^T)=n-1$ on a connected graph with $n$ nodes. Each leverage depends only on the graph and its conductances. The actual current in a specified operating condition also depends on the chosen source and sink.

## Solve the triangle by hand

With 1 S on every branch, the detour A–C–B consists of two 1 Ω resistors in series. Inject 1 A at A and extract it at B. The detour's 0.5 S combines with the direct 1 S, so the measured drop is $1/1.5=2/3$ V. The direct current is 2/3 A. The dead-end C–D branch carries zero current in this experiment. When tested between its own endpoints, however, C–D carries the entire test current. The four leverages are consequently $2/3,2/3,2/3,1$ and sum to three.

Double A–B's conductance to 2 S. Its leverage rises to $2/2.5=0.8$, while the voltage needed for the same 1 A drops to 0.4 V. This is why the two traces in the experiment have opposite slopes. Rerun with a weaker bridge. Its resistance increases, yet its leverage stays equal to one.

## Set the experimental inputs

Edit these values, then run the calculation and both figures again.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

teal, orange, blue = "#19596a", "#aa462a", "#3868a0"
def axes(xlabel, ylabel):
    fig, ax = plt.subplots(figsize=(7, 4), constrained_layout=True)
    ax.set(xlabel=xlabel, ylabel=ylabel)
    ax.grid(alpha=0.2)
    return fig, ax

In [ ]:
direct_siemens = 1.0
bridge_siemens = 1.0

In [ ]:
assert direct_siemens > 0 and bridge_siemens > 0
edges = [(0, 1), (1, 2), (2, 0), (2, 3)]
B = np.zeros((4, 4))
for col, (i, j) in enumerate(edges):
    B[i, col], B[j, col] = 1, -1
grounded = B[:-1]

def network(g):
    weights = np.array([g, 1., 1., bridge_siemens])
    L = (grounded * weights) @ grounded.T
    voltage = np.linalg.solve(L, grounded)
    R = np.sum(grounded * voltage, axis=0)
    return R, weights * R

resistance, leverage = network(direct_siemens)
assert np.isclose(leverage.sum(), 3) and np.isclose(leverage[-1], 1)
assert np.isclose(resistance[0], 1/(direct_siemens + .5))
print("Edge    resistance (ohm)    leverage")
for label, r, share in zip(["AB","BC","CA","CD"], resistance, leverage):
    print(f"{label:3} {r:13.5f} {share:12.5f}")

In [ ]:
coords = np.array([[-1.,0],[1.,0],[0,1.3],[0,2.25]])
fig, ax = axes("Network position", "Network position")
for (i,j), share in zip(edges, leverage):
    ends = coords[[i,j]]
    ax.plot(*ends.T, color=teal, lw=1.5+3*share, zorder=1)
    center = ends.mean(axis=0)
    ax.text(*center, f"{share:.3f}", ha="center", va="center",
            bbox=dict(facecolor="white", edgecolor="none", pad=2))
for (x,y), label in zip(coords,"ABCD"):
    ax.scatter(x,y,s=440,color=blue,zorder=3)
    ax.text(x,y,label,ha="center",va="center",color="white",zorder=4)
ax.set_aspect("equal"); ax.set_xlim(-1.35,1.35); ax.set_ylim(-.3,2.56)
ax.axis("off")
plt.show()

Each label gives the share of a separate one-ampere test between that edge's endpoints. The pendant branch remains at unity for any positive conductance.

In [ ]:
sweep = np.geomspace(min(.05, direct_siemens),max(20, direct_siemens),120)
result = [network(g) for g in sweep]
fig, ax = axes("Direct conductance AB (S)", "Current share (dimensionless)")
ax.semilogx(sweep, [s[0] for r,s in result], color=teal, label="AB current share")
ax.scatter(direct_siemens, leverage[0], color=teal)
right = ax.twinx(); right.set_ylabel("Effective resistance AB (Ω)")
right.plot(sweep, [r[0] for r,s in result], color=orange, label="Effective resistance")
right.scatter(direct_siemens, resistance[0], color=orange)
ax.legend(loc="center left"); right.legend(loc="center right")
plt.show()

The current-share curve tends to one as the direct branch dominates the detour. The voltage per ampere falls from two ohms toward zero.

## Exercise · remove the direct branch

How much does the terminal resistance rise when the 1 S direct edge is removed? What changes if the bridge is removed instead?

For A and B the sole remaining connection consists of two 1 Ω resistors, so resistance rises from 2/3 Ω to 2 Ω, an increase of 4/3 Ω. Removing C–D instead isolates D. Measurements within the triangle remain finite, but a test injecting current at D and withdrawing it at C no longer has a finite DC voltage solution.

## Exercise · differentiate the measured voltage

Under a fixed one-ampere A-to-B test, find the sensitivity of the terminal resistance to the direct conductance at 1 S.

For variable $g$ the resistance is $1/(g+0.5)$. Differentiation gives $-1/(g+0.5)^2=-4/9$ Ω/S at the chosen point. The magnitude equals the square of the voltage drop per ampere across that branch. Differentiating $L_gv=f$ at fixed injection gives the same result for any branch, $\partial (f^Tv)/\partial g_e=-(b_e^Tv)^2$ for the unit test.